In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
    .appName("orders")\
    .getOrCreate()   


data = [
    (1,"ravi",100),
    (2,"srinu",200),
    (3,"pavan",300),
    (4,"praveen",400),
    (5,"sai",450),
    (6,"shiva",500),
    (7,"srinu",200),
    (8,"praveen",400),
    (9,"sai",450),
    (10,"shiva",500),
    (11,"srinu",200),
    (12,"praveen",400),
    (13,"sai",450)
]

columns = ["emp_id","name","salary"]

employees = spark.createDataFrame(data,columns)

employees.display()

emp_id,name,salary
1,ravi,100
2,srinu,200
3,pavan,300
4,praveen,400
5,sai,450
6,shiva,500
7,srinu,200
8,praveen,400
9,sai,450
10,shiva,500


In [0]:
employees.printSchema()

employees.show()

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)

+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     1|   ravi|   100|
|     2|  srinu|   200|
|     3|  pavan|   300|
|     4|praveen|   400|
|     5|    sai|   450|
|     6|  shiva|   500|
|     7|  srinu|   200|
|     8|praveen|   400|
|     9|    sai|   450|
|    10|  shiva|   500|
|    11|  srinu|   200|
|    12|praveen|   400|
|    13|    sai|   450|
+------+-------+------+



In [0]:
df_csv = spark.read\
    .option("header",True)\
    .option("inferschema",True)\
    .csv("/Volumes/yaswanth/default/data/datasets/datasets/customers.csv") 
    
df_csv.display()       

customer_id,first_name,last_name,city,state,signup_date
1,Aarav,Nair,Bengaluru,KA,2024-09-07
2,Vikram,Reddy,Mumbai,MH,2025-07-12
3,Varun,Rao,Bengaluru,KA,2024-02-02
4,Rohan,Patel,Hyderabad,TG,2024-08-26
5,Neha,Sharma,Mysuru,KA,2025-07-28
6,Riya,Rao,Mumbai,MH,2024-08-13
7,Varun,Nair,Visakhapatnam,AP,2024-01-07
8,Ishita,Gupta,Chennai,TN,2024-10-11
9,Arjun,Gupta,Chennai,TN,2024-04-14
10,Siddharth,Reddy,Bengaluru,KA,2025-01-02


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import(
    col,row_number,rank,dense_rank,
    lag,lead,ntile,sum,count,max,min
)

data = [
    (101, "Ravi",   "IT",    80000),
    (102, "Priya",  "IT",    95000),
    (103, "Kumar",  "IT",    95000),
    (104, "Sita",   "HR",    60000),
    (105, "Arjun",  "HR",    75000),
    (106, "Divya",  "HR",    50000),
    (107, "Kiran",  "Sales", 70000),
    (108, "Meena",  "Sales", 90000),
    (109, "Rahul",  "Sales", 65000),
    (110, "Anu",    "IT",    60000)
]

columns = ["emp_id", "name", "department", "salary"]

employees = spark.createDataFrame(data, columns)

display(employees)

emp_id,name,department,salary
101,Ravi,IT,80000
102,Priya,IT,95000
103,Kumar,IT,95000
104,Sita,HR,60000
105,Arjun,HR,75000
106,Divya,HR,50000
107,Kiran,Sales,70000
108,Meena,Sales,90000
109,Rahul,Sales,65000
110,Anu,IT,60000


In [0]:
window_spec = Window.partitionBy("department").orderBy(F.col("salary").desc())

df_all = employees.withColumn("row_number", row_number().over(window_spec))\
    .withColumn("rank", rank().over(window_spec))\
    .withColumn("dense_rank", dense_rank().over(window_spec))\
    .withColumn("lag", lag("salary", 1).over(window_spec))\
    .withColumn("lead", lead("salary", 1).over(window_spec))\
    .withColumn("ntile", ntile(4).over(window_spec))

display(df_all)


emp_id,name,department,salary,row_number,rank,dense_rank,lag,lead,ntile
105,Arjun,HR,75000,1,1,1,null,60000,1
104,Sita,HR,60000,2,2,2,75000,50000,2
106,Divya,HR,50000,3,3,3,60000,null,3
102,Priya,IT,95000,1,1,1,null,95000,1
103,Kumar,IT,95000,2,1,1,95000,80000,2
101,Ravi,IT,80000,3,3,2,95000,60000,3
110,Anu,IT,60000,4,4,3,80000,null,4
108,Meena,Sales,90000,1,1,1,null,70000,1
107,Kiran,Sales,70000,2,2,2,90000,65000,2
109,Rahul,Sales,65000,3,3,3,70000,null,3


In [0]:
df = employees.filter(F.col("department")== "IT").agg(F.avg("salary")).show()

+-----------+
|avg(salary)|
+-----------+
|    82500.0|
+-----------+



In [0]:
windowSpec = Window.partitionBy("department").orderBy(F.col("salary").desc())

df_all = employees.withColumn("avg_salary" ,F.avg('salary').over(windowSpec))

display(df_all)

ranking = employees.withColumn("ranking", F.rank().over(windowSpec))
display(ranking)



emp_id,name,department,salary,avg_salary
105,Arjun,HR,75000,75000.0
104,Sita,HR,60000,67500.0
106,Divya,HR,50000,61666.666666666664
102,Priya,IT,95000,95000.0
103,Kumar,IT,95000,95000.0
101,Ravi,IT,80000,90000.0
110,Anu,IT,60000,82500.0
108,Meena,Sales,90000,90000.0
107,Kiran,Sales,70000,80000.0
109,Rahul,Sales,65000,75000.0


emp_id,name,department,salary,ranking
105,Arjun,HR,75000,1
104,Sita,HR,60000,2
106,Divya,HR,50000,3
102,Priya,IT,95000,1
103,Kumar,IT,95000,1
101,Ravi,IT,80000,3
110,Anu,IT,60000,4
108,Meena,Sales,90000,1
107,Kiran,Sales,70000,2
109,Rahul,Sales,65000,3


In [0]:
windowSpec = Window.partitionBy("department").orderBy(F.col("salary").desc())

avg_dept_salary = employees.withColumn("avg_dept_salary", F.avg("salary").over(windowSpec))

display(avg_dept_salary)

salary_rank = employees.withColumn("salary_rank", F.rank().over(windowSpec))

display(salary_rank)
 
fil = avg_dept_salary.filter(F.col("salary") > F.col("avg_dept_salary"))
display(fil)

fil2 = salary_rank.filter(F.col("salary_rank") <= 2)

display(fil2)

emp_id,name,department,salary,avg_dept_salary
105,Arjun,HR,75000,75000.0
104,Sita,HR,60000,67500.0
106,Divya,HR,50000,61666.666666666664
102,Priya,IT,95000,95000.0
103,Kumar,IT,95000,95000.0
101,Ravi,IT,80000,90000.0
110,Anu,IT,60000,82500.0
108,Meena,Sales,90000,90000.0
107,Kiran,Sales,70000,80000.0
109,Rahul,Sales,65000,75000.0


emp_id,name,department,salary,salary_rank
105,Arjun,HR,75000,1
104,Sita,HR,60000,2
106,Divya,HR,50000,3
102,Priya,IT,95000,1
103,Kumar,IT,95000,1
101,Ravi,IT,80000,3
110,Anu,IT,60000,4
108,Meena,Sales,90000,1
107,Kiran,Sales,70000,2
109,Rahul,Sales,65000,3


emp_id,name,department,salary,avg_dept_salary


emp_id,name,department,salary,salary_rank
105,Arjun,HR,75000,1
104,Sita,HR,60000,2
102,Priya,IT,95000,1
103,Kumar,IT,95000,1
108,Meena,Sales,90000,1
107,Kiran,Sales,70000,2
